In [28]:
import pandas as pd

df = pd.read_excel("2022ar_quadro_resultados.xlsx")

print(df.head(10))

       Unnamed: 0 Unnamed: 1 Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5  \
0         Círculo        NaN          1          2          3          4   
1             NaN        NaN     Aveiro       Beja      Braga   Bragança   
2       Inscritos        NaN     642623     120204     776543     137572   
3  Votantes (VTT)     Número     364875      67520     494529      65738   
4             NaN   % / INSC      56.78      56.17      63.68      47.78   
5       Abstenção   % / INSC      43.22      43.83      36.32      52.22   
6         Brancos     Número       4743        763       6158        567   
7             NaN    % / VTT        1.3       1.13       1.25       0.86   
8           Nulos     Número       3425        634       3846        795   
9             NaN    % / VTT       0.94       0.94       0.78       1.21   

       Unnamed: 6 Unnamed: 7 Unnamed: 8 Unnamed: 9  ... Unnamed: 15  \
0               5          6          7          8  ...          14   
1  Castelo Branco    

In [29]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="portugal_elections",
    user="postgres",
    password="C1w3v318"
)

cur = conn.cursor()

In [ ]:
results = []

districts = df.iloc[1, 2:].tolist()

i = 12

while i + 2 < len(df):

    party = df.iloc[i, 0]

    if pd.isna(party):
        i += 1
        continue

    votes_row = df.iloc[i]
    percentage_row = df.iloc[i + 1]
    mandates_row = df.iloc[i + 2]

    for col in range(2, len(df.columns)):

        district = districts[col - 2]

        votes = votes_row.iloc[col]
        percentage = percentage_row.iloc[col]
        mandates = mandates_row.iloc[col]

        results.append({
            "district": district,
            "party": party,
            "votes": votes,
            "vote_percentage": percentage,
            "mandates": mandates
        })

    i += 3

clean_df = pd.DataFrame(results)

print(clean_df.head(10))

         district party votes vote_percentage mandates
0          Aveiro     A     -               -        -
1            Beja     A     -               -        -
2           Braga     A   286            0.07        0
3        Bragança     A     -               -        -
4  Castelo Branco     A     -               -        -
5         Coimbra     A   320            0.15        0
6           Évora     A     -               -        -
7            Faro     A     -               -        -
8          Guarda     A     -               -        -
9          Leiria     A     -               -        -


In [ ]:
# Raw Results ETL

cur.execute("""
    TRUNCATE TABLE staging.raw_results
    RESTART IDENTITY CASCADE;
""")

for _, row in clean_df.iterrows():

    cur.execute("""
        INSERT INTO staging.raw_results (
            district,
            party,
            votes,
            vote_percentage,
            mandates
        )
        VALUES (%s, %s, %s, %s, %s)
    """, (
        row["district"],
        row["party"],
        row["votes"],
        row["vote_percentage"],
        row["mandates"]
    ))

conn.commit()

In [32]:
# Clean DataFrame

clean_df = clean_df.replace("-", None)

clean_df = clean_df[
    ~clean_df["party"].astype(str).str.contains("mandatos", na=False)
]

clean_df["votes"] = pd.to_numeric(
    clean_df["votes"]
)

clean_df["vote_percentage"] = pd.to_numeric(
    clean_df["vote_percentage"]
)

clean_df["mandates"] = pd.to_numeric(
    clean_df["mandates"]
)

clean_df = clean_df.dropna(
    subset=["votes", "vote_percentage", "mandates", "district"]
)

print(clean_df.dtypes)
print(clean_df.head(20))

clean_df.to_csv("clean_results.csv", index=False)

district               str
party                  str
votes              float64
vote_percentage    float64
mandates           float64
dtype: object
            district party    votes  vote_percentage  mandates
2              Braga     A    286.0             0.07       0.0
5            Coimbra     A    320.0             0.15       0.0
10            Lisboa     A    747.0             0.06       0.0
12             Porto     A    378.0             0.04       0.0
15  Viana do Castelo     A    173.0             0.14       0.0
20            Europa     A     84.0             0.11       0.0
21    Fora da Europa     A    479.0             0.77       0.0
23            Aveiro   ADN    677.0             0.19       0.0
27    Castelo Branco   ADN    176.0             0.19       0.0
30              Faro   ADN   1157.0             0.61       0.0
32            Leiria   ADN    783.0             0.34       0.0
33            Lisboa   ADN   3208.0             0.28       0.0
35             Porto   ADN   143

In [33]:
# ETL districts

districts_df = pd.DataFrame({
    "name": clean_df["district"].unique()
})

cur.execute("""
    TRUNCATE TABLE districts RESTART IDENTITY CASCADE;
""")

for _, row in districts_df.iterrows():

    cur.execute("""
        INSERT INTO districts (name)
        VALUES (%s)
    """, (row["name"],))

conn.commit()

In [34]:
# ETL parties

parties_df = pd.DataFrame({
    "acronym": clean_df["party"].unique()
})

cur.execute("""
    TRUNCATE TABLE parties RESTART IDENTITY CASCADE;
""")

for _, row in parties_df.iterrows():

    cur.execute("""
        INSERT INTO parties (acronym, full_name)
        VALUES (%s, %s)
    """, (
        row["acronym"],
        row["acronym"]
    ))

conn.commit()

In [35]:
cur.execute("SELECT district_id, name FROM districts")

district_lookup = {}

for district_id, name in cur.fetchall():
    district_lookup[name] = district_id

print(district_lookup)

{'Braga': 1, 'Coimbra': 2, 'Lisboa': 3, 'Porto': 4, 'Viana do Castelo': 5, 'Europa': 6, 'Fora da Europa': 7, 'Aveiro': 8, 'Castelo Branco': 9, 'Faro': 10, 'Leiria': 11, 'Setúbal': 12, 'Viseu': 13, 'RA Açores': 14, 'RA Madeira': 15, 'Beja': 16, 'Bragança': 17, 'Évora': 18, 'Guarda': 19, 'Portalegre': 20, 'Santarém': 21, 'Vila Real': 22}


In [36]:
cur.execute("SELECT party_id, acronym FROM parties")

party_lookup = {}

for party_id, acronym in cur.fetchall():
    party_lookup[acronym] = party_id

print(party_lookup)

{'A': 1, 'ADN': 2, 'B.E.': 3, 'CDS-PP': 4, 'CH': 5, 'E': 6, 'IL': 7, 'JPP': 8, 'L': 9, 'MAS': 10, 'MPT': 11, 'NC': 12, 'PAN': 13, 'PCP-PEV': 14, 'PCTP/MRPP': 15, 'PPD/PSD': 16, 'PPD/PSD.CDS-PP': 17, 'PPD/PSD.CDS-PP.PPM': 18, 'PPM': 19, 'PS': 20, 'PTP': 21, 'R.I.R.': 22, 'VP': 23}


In [ ]:
# ETL Results

cur.execute("""
    TRUNCATE TABLE results RESTART IDENTITY CASCADE;
""")

for _, row in clean_df.iterrows():

    district_id = district_lookup[row["district"]]
    party_id = party_lookup[row["party"]]
    cur.execute("""
        INSERT INTO results (
            election_id,
            district_id,
            party_id,
            votes,
            vote_percentage,
            mandates
        )
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        1,
        district_id,
        party_id,
        row["votes"],
        row["vote_percentage"],
        row["mandates"]
    ))

conn.commit()

In [ ]:
# ETL offices

cur.execute("""
    TRUNCATE TABLE offices RESTART IDENTITY CASCADE;
""")

cur.execute("""
    INSERT INTO offices (
        name,
        description
    )
    VALUES (%s, %s)
""", (
    "Portuguese Parliament",
    "Parliamentary elections in Portugal"
))

conn.commit()

In [ ]:
# ETL coalitions

cur.execute("""
    TRUNCATE TABLE coalitions RESTART IDENTITY CASCADE;
""")

coalitions_df = parties_df[
    parties_df["acronym"].str.contains("/", na=False)
]

for _, row in coalitions_df.iterrows():

    cur.execute("""
        INSERT INTO coalitions (
            acronym,
            full_name
        )
        VALUES (%s, %s)
    """, (
        row["acronym"],
        row["acronym"]
    ))

conn.commit()

In [40]:
cur.execute("""
    SELECT coalition_id, acronym
    FROM coalitions
""")

coalition_lookup = {}

for coalition_id, acronym in cur.fetchall():

    coalition_lookup[acronym] = coalition_id

In [ ]:
# ETL candidacies

cur.execute("""
    TRUNCATE TABLE candidacies RESTART IDENTITY CASCADE;
""")

candidacies_df = clean_df[
    ["district", "party"]
].drop_duplicates()

for _, row in candidacies_df.iterrows():

    district_id = district_lookup[row["district"]]

    party_id = party_lookup[row["party"]]

    coalition_id = None

    if row["party"] in coalition_lookup:

        coalition_id = coalition_lookup[row["party"]]

    cur.execute("""
        INSERT INTO candidacies (
            election_id,
            district_id,
            party_id,
            coalition_id,
            candidate_name
        )
        VALUES (%s, %s, %s, %s, %s)
    """, (
        1,
        district_id,
        party_id,
        coalition_id,
        None
    ))

conn.commit()

In [42]:
participation = []
districts = df.iloc[1, 2:].tolist()

registered_row = df.iloc[2]
voters_row = df.iloc[3]
abstention_row = df.iloc[5]
blank_row = df.iloc[6]
null_row = df.iloc[8]
valid_row = df.iloc[10]

for col in range(2, len(df.columns)):

    district = districts[col - 2]

    participation.append({
        "district": district,

        "registered_voters": registered_row.iloc[col],
        "voters": voters_row.iloc[col],

        "abstention_percentage": abstention_row.iloc[col],

        "blank_votes": blank_row.iloc[col],
        "null_votes": null_row.iloc[col],

        "valid_votes": valid_row.iloc[col]
    })

participation_df = pd.DataFrame(participation)

print(participation_df.head())

         district  registered_voters  voters  abstention_percentage  \
0          Aveiro             642623  364875                  43.22   
1            Beja             120204   67520                  43.83   
2           Braga             776543  494529                  36.32   
3        Bragança             137572   65738                  52.22   
4  Castelo Branco             166269   95738                  42.42   

   blank_votes  null_votes  valid_votes  
0         4743        3425       356707  
1          763         634        66123  
2         6158        3846       484525  
3          567         795        64376  
4         1117        1126        93495  


In [43]:
participation_df = participation_df[
    participation_df["district"] != "Total"
]

numeric_columns = [
    "registered_voters",
    "voters",
    "abstention_percentage",
    "blank_votes",
    "null_votes",
    "valid_votes"
]

for col in numeric_columns:

    participation_df[col] = pd.to_numeric(
        participation_df[col],
        errors="coerce"
    )

In [44]:
print(participation_df.dtypes)
print(participation_df.head())

district                     str
registered_voters          int64
voters                     int64
abstention_percentage    float64
blank_votes                int64
null_votes                 int64
valid_votes                int64
dtype: object
         district  registered_voters  voters  abstention_percentage  \
0          Aveiro             642623  364875                  43.22   
1            Beja             120204   67520                  43.83   
2           Braga             776543  494529                  36.32   
3        Bragança             137572   65738                  52.22   
4  Castelo Branco             166269   95738                  42.42   

   blank_votes  null_votes  valid_votes  
0         4743        3425       356707  
1          763         634        66123  
2         6158        3846       484525  
3          567         795        64376  
4         1117        1126        93495  


In [ ]:
# ETL participation

participation_df = participation_df.dropna(
    subset=[
        "district",
        "registered_voters",
        "voters"
    ]
)

cur.execute("""
    TRUNCATE TABLE participation RESTART IDENTITY;
""")

for _, row in participation_df.iterrows():

    district_id = district_lookup[row["district"]]

    cur.execute("""
        INSERT INTO participation (
            election_id,
            district_id,

            registered_voters,
            voters,
            abstention_percentage,

            blank_votes,
            null_votes,
            valid_votes
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        1,
        district_id,

        row["registered_voters"],
        row["voters"],
        row["abstention_percentage"],

        row["blank_votes"],
        row["null_votes"],
        row["valid_votes"]
    ))

conn.commit()

In [46]:
cur.close()
conn.close()